In [ ]:
## layer 2

In [ ]:
import torch 


# input of layer 2
bs = 16 # batch size

Leye = torch.rand(bs,128, 8, 8)
print(Leye)
Reye = torch.rand(bs,128, 8, 8)
print(Reye)
FaceData = torch.rand(bs,128, 8, 8)  # currently matched with eyes......
print(FaceData)

class TinyModel(torch.nn.Module):
    def __init__(self):
        super(TinyModel, self).__init__()
        # layer 1: feature extraction (to be implemented)
        
        
        # layer 2: feature fusion: concate + group  normalization
        # self.concate = torch.cat((x, x, x), 0) // not needed here, only in forward
        # total_ch = Leye[1]+Reye[1]+FaceData[1]  # total channels of the input // this is not working 
        total_ch = 384
        self.gn = torch.nn.GroupNorm(3, total_ch)     # Separate 6 channels into 3 groups, how to define a appropriate number of groups & channels?
        
        # layer 3:self-attention
        # self.norm = torch.nn.LayerNorm([total_ch, 8, 8])  # here is the problem!!! 
        self.norm = torch.nn.LayerNorm(total_ch)
        self.mha = torch.nn.MultiheadAttention(total_ch, num_heads=1, batch_first=True)
        self.scale = torch.nn.Parameter(torch.zeros(1))

    def use_attention(self, x):
        # Reshape input for multi-head attention
        bs, c, h, w = x.shape
        x_att = x.reshape(bs, c, h * w).transpose(1, 2)  # BSxHWxC
        
        # Apply Layer Normalization
        x_att = self.norm(x_att)
        Q = K = V = x_att
        # Apply Multi-head Attention
        att_out, att_map  = self.mha(Q, K, V)  # Q,K,V  Q=K=V, self-attention
        return att_out.transpose(1, 2).reshape(bs, c, h, w), att_map # transpose first, then Reshape output to original shape (bs, c, h, w)


    def forward(self, left_eye, right_eye, face):

        # layer2
        concate = torch.cat((left_eye, right_eye, face), 1)  # dim = 0 or 1?  only channel dim changes?
        out = self.gn(concate)
        # out = out.unsqueeze(bs)  
        att_out, _ = self.use_attention(out)
        out = out + self.scale * att_out
        return out 
        

model = TinyModel()
print(model)

output = model(Leye, Reye, FaceData)  # currently the face Data is not matched with eyes, should be matched in the future, how to match? linear padding or other methods? would it affect the performance if change the size of facedata
print("output:",output.shape)


tensor([[[[6.3694e-01, 5.8495e-01, 6.8840e-01,  ..., 3.2761e-01,
           1.3103e-01, 7.3556e-01],
          [7.1565e-01, 4.2686e-01, 1.4433e-01,  ..., 1.8247e-01,
           1.5647e-02, 9.4672e-01],
          [6.2590e-01, 7.8904e-01, 5.0826e-01,  ..., 6.5074e-01,
           6.8072e-01, 9.9535e-01],
          ...,
          [7.7962e-01, 9.0747e-01, 5.9768e-01,  ..., 9.3773e-01,
           1.2458e-01, 5.1368e-01],
          [6.7121e-01, 3.1341e-01, 6.8852e-01,  ..., 2.5422e-01,
           1.4815e-01, 9.9416e-01],
          [4.5799e-01, 3.6575e-01, 3.1670e-01,  ..., 6.8717e-01,
           4.0176e-01, 6.7444e-03]],

         [[8.3616e-01, 1.2066e-01, 3.0784e-01,  ..., 5.3300e-01,
           7.5992e-01, 9.6179e-01],
          [1.2912e-01, 7.9402e-01, 5.4576e-01,  ..., 6.4014e-01,
           2.8674e-01, 9.5681e-01],
          [6.0169e-01, 6.8479e-01, 4.5292e-01,  ..., 2.1732e-01,
           7.0414e-01, 4.8884e-02],
          ...,
          [5.5835e-01, 2.0319e-01, 9.1396e-01,  ..., 7.1939